# 🐱 Cat Detector 9000™
### A Neural Network That Finally Understands Cats Better Than I Do

> **"Is it a cat?"** — Humanity's most important unanswered question.

**Architecture:** `Image → Flatten → Hidden (ReLU) → Hidden (ReLU) → Sigmoid → 🐱 / 🚫`

---

## 📋 Table of Contents
1. [Imports](#1-imports)
2. [Load Dataset](#2-load-dataset)
3. [Preprocessing & Visualization](#3-preprocessing--visualization)
4. [Neural Network (Pure NumPy)](#4-neural-network-pure-numpy)
5. [Training](#5-training)
6. [Training Curves](#6-training-curves)
7. [Evaluation](#7-evaluation)
8. [Predict Your Own Image](#8-predict-your-own-image)
9. [Hyperparameter Experiments](#9-hyperparameter-experiments)

---
## 1. Imports

In [ ]:
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
from google.colab import files

try:
    import h5py
except ImportError:
    !pip install -q h5py
    import h5py

print('✅ All libraries imported successfully!')

---
## 2. Load Dataset

In [ ]:
# upload the h5 dataset files from your local machine
print('Upload train_catvnoncat.h5')
uploaded = files.upload()

print('Upload test_catvnoncat.h5')
uploaded2 = files.upload()

In [ ]:
# read train set from h5 file
with h5py.File('train_catvnoncat.h5', 'r') as f:
    train_x_orig = np.array(f['train_set_x'][:])
    train_y      = np.array(f['train_set_y'][:]).reshape(1, -1)
    classes      = np.array(f['list_classes'][:])

# read test set from h5 file
with h5py.File('test_catvnoncat.h5', 'r') as f:
    test_x_orig = np.array(f['test_set_x'][:])
    test_y      = np.array(f['test_set_y'][:])

print('✅ Datasets loaded successfully')
print(f'Train X : {train_x_orig.shape}')
print(f'Train Y : {train_y.shape}')
print(f'Test  X : {test_x_orig.shape}')
print(f'Test  Y : {test_y.shape}')
print(f'Classes : {[c.decode() for c in classes]}')
print(f'Cat ratio: {train_y.mean():.1%}')

---
## 3. Preprocessing & Visualization

In [ ]:
# flatten (H, W, C) to a single vector per image, then normalise pixels to [0, 1]
m_train = train_x_orig.shape[0]
m_test  = test_x_orig.shape[0]

train_x = train_x_orig.reshape(m_train, -1).T / 255.0
test_x  = test_x_orig.reshape(m_test, -1).T  / 255.0

print(f'train_x : {train_x.shape}  (12288 = 64x64x3 pixels per image)')
print(f'test_x  : {test_x.shape}')
print(f'Pixel range: [{train_x.min():.2f}, {train_x.max():.2f}]')

In [ ]:
# show 6 cat samples (top row) and 6 non-cat samples (bottom row)
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Sample Training Images — Cat Detector 9000™',
             color='white', fontsize=13, fontweight='bold')

cat_idx    = np.where(train_y[0] == 1)[0][:6]
noncat_idx = np.where(train_y[0] == 0)[0][:6]

for i, idx in enumerate(cat_idx):
    axes[0, i].imshow(train_x_orig[idx])
    axes[0, i].set_title('CAT', color='#4ade80', fontsize=10)
    axes[0, i].axis('off')

for i, idx in enumerate(noncat_idx):
    axes[1, i].imshow(train_x_orig[idx])
    axes[1, i].set_title('NOT CAT', color='#f87171', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

---
## 4. Neural Network (Pure NumPy)

Full forward pass, cost, backprop, and weight update — implemented from scratch using only NumPy.

In [ ]:
# activation functions

def sigmoid(z):
    # clip prevents overflow in exp for very large/small z
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    return np.maximum(0, z)

def relu_backward(dA, z):
    # gradient is 0 where z was <= 0 (dead neuron), 1 elsewhere
    dZ = dA.copy()
    dZ[z <= 0] = 0
    return dZ

In [ ]:
def init_params(dims):
    """
    He initialisation for each layer.
    dims: list of layer sizes, e.g. [12288, 64, 32, 1]
    """
    np.random.seed(42)
    params = {}
    for l in range(1, len(dims)):
        params[f'W{l}'] = np.random.randn(dims[l], dims[l-1]) * np.sqrt(2 / dims[l-1])
        params[f'b{l}'] = np.zeros((dims[l], 1))
    return params


def forward(X, params):
    """
    Forward pass: ReLU for all hidden layers, Sigmoid for output layer.
    Returns final activation AL and cache of intermediates for backprop.
    """
    cache = {'A0': X}
    L = len(params) // 2
    A = X

    # hidden layers with ReLU
    for l in range(1, L):
        Z = params[f'W{l}'] @ A + params[f'b{l}']
        A = relu(Z)
        cache[f'Z{l}'] = Z
        cache[f'A{l}'] = A

    # output layer with Sigmoid
    Z = params[f'W{L}'] @ A + params[f'b{L}']
    AL = sigmoid(Z)
    cache[f'Z{L}'] = Z
    cache[f'A{L}'] = AL

    return AL, cache


def compute_cost(AL, Y):
    """Binary cross-entropy loss."""
    m   = Y.shape[1]
    eps = 1e-8  # avoid log(0)
    return float(-np.sum(Y * np.log(AL + eps) + (1 - Y) * np.log(1 - AL + eps)) / m)


def backward(AL, Y, params, cache):
    """
    Backpropagation: computes gradients for all W and b.
    Output layer uses sigmoid derivative; hidden layers use relu_backward.
    """
    m     = Y.shape[1]
    L     = len(params) // 2
    grads = {}

    dA = -(Y / (AL + 1e-8)) + (1 - Y) / (1 - AL + 1e-8)

    # output layer gradients
    s  = sigmoid(cache[f'Z{L}'])
    dZ = dA * s * (1 - s)
    grads[f'dW{L}'] = (dZ @ cache[f'A{L-1}'].T) / m
    grads[f'db{L}'] = np.sum(dZ, axis=1, keepdims=True) / m
    dA = params[f'W{L}'].T @ dZ

    # hidden layer gradients (in reverse)
    for l in reversed(range(1, L)):
        dZ = relu_backward(dA, cache[f'Z{l}'])
        grads[f'dW{l}'] = (dZ @ cache[f'A{l-1}'].T) / m
        grads[f'db{l}'] = np.sum(dZ, axis=1, keepdims=True) / m
        dA = params[f'W{l}'].T @ dZ

    return grads


def update(params, grads, lr):
    """Vanilla gradient descent parameter update."""
    p = copy.deepcopy(params)
    for l in range(1, len(params) // 2 + 1):
        p[f'W{l}'] -= lr * grads[f'dW{l}']
        p[f'b{l}'] -= lr * grads[f'db{l}']
    return p


def accuracy(AL, Y):
    """Returns accuracy as a percentage."""
    return float(np.mean((AL > 0.5).astype(int) == Y)) * 100


print('✅ Neural network functions defined: sigmoid, relu, forward, compute_cost, backward, update')

---
## 5. Training

In [ ]:
# architecture: [12288 input] -> [64 ReLU] -> [32 ReLU] -> [1 Sigmoid]
LAYER_DIMS    = [train_x.shape[0], 64, 32, 1]
LEARNING_RATE = 0.0075
NUM_ITERS     = 2500
PRINT_EVERY   = 250

params = init_params(LAYER_DIMS)
costs, train_accs, test_accs = [], [], []

print('Training Cat Detector 9000™')
print(f'  Architecture  : {LAYER_DIMS}')
print(f'  Learning rate : {LEARNING_RATE}')
print(f'  Iterations    : {NUM_ITERS}')
print('-' * 55)

t0 = time.time()
for i in range(NUM_ITERS):
    AL, cache = forward(train_x, params)
    cost      = compute_cost(AL, train_y)
    grads     = backward(AL, train_y, params, cache)
    params    = update(params, grads, LEARNING_RATE)

    if i % PRINT_EVERY == 0:
        tr_acc  = accuracy(AL, train_y)
        AL_t, _ = forward(test_x, params)
        te_acc  = accuracy(AL_t, test_y)
        costs.append(cost)
        train_accs.append(tr_acc)
        test_accs.append(te_acc)
        emoji = '🐱' if te_acc > 65 else '🤔'
        print(f'  Iter {i:4d} | Loss: {cost:.4f} | Train: {tr_acc:5.1f}% | Test: {te_acc:5.1f}% {emoji}')

print('-' * 55)
print(f'Done in {time.time()-t0:.1f}s!')

---
## 6. Training Curves

In [ ]:
iters = list(range(0, NUM_ITERS, PRINT_EVERY))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

for ax in (ax1, ax2):
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#9ca3af')
    for sp in ax.spines.values():
        sp.set_color('#374151')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.2, color='#374151')

ax1.plot(iters, costs, color='#f59e0b', lw=2.5, marker='o', ms=5)
ax1.fill_between(iters, costs, alpha=0.2, color='#f59e0b')
ax1.set_title('Loss', color='white', fontsize=12)
ax1.set_xlabel('Iteration', color='#9ca3af')
ax1.set_ylabel('Cost', color='#9ca3af')

ax2.plot(iters, train_accs, color='#4ade80', lw=2.5, marker='o', ms=5, label='Train')
ax2.plot(iters, test_accs,  color='#60a5fa', lw=2.5, marker='s', ms=5, label='Test')
ax2.axhline(50, color='#f87171', ls='--', alpha=0.5, label='Random (50%)')
ax2.set_title('Accuracy', color='white', fontsize=12)
ax2.set_xlabel('Iteration', color='#9ca3af')
ax2.set_ylabel('Accuracy (%)', color='#9ca3af')
ax2.legend(facecolor='#0f0f1a', labelcolor='white')
ax2.set_ylim(0, 105)

plt.suptitle('Cat Detector 9000™ — Training Results', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Evaluation

In [ ]:
def evaluate(X, Y, params, name):
    """
    Computes accuracy, precision, recall, F1, and confusion matrix values
    for a given split.
    """
    AL, _ = forward(X, params)
    preds  = (AL > 0.5).astype(int)

    TP = int(np.sum((preds == 1) & (Y == 1)))
    TN = int(np.sum((preds == 0) & (Y == 0)))
    FP = int(np.sum((preds == 1) & (Y == 0)))
    FN = int(np.sum((preds == 0) & (Y == 1)))

    acc  = (TP + TN) / Y.shape[1] * 100
    prec = TP / (TP + FP + 1e-8) * 100
    rec  = TP / (TP + FN + 1e-8) * 100
    f1   = 2 * prec * rec / (prec + rec + 1e-8)

    print(f'\n=== {name} ===')
    print(f'  Accuracy  : {acc:.1f}%')
    print(f'  Precision : {prec:.1f}%')
    print(f'  Recall    : {rec:.1f}%')
    print(f'  F1 Score  : {f1:.1f}%')
    print(f'  Confusion : TP={TP} TN={TN} FP={FP} FN={FN}')
    return preds, AL


train_preds, train_probs = evaluate(train_x, train_y, params, 'TRAIN SET')
test_preds,  test_probs  = evaluate(test_x,  test_y,  params, 'TEST SET')

---
## 8. Predict Your Own Image

In [ ]:
def predict_single(img_array, params):
    """Resize image to 64x64, run forward pass, and display verdict with confidence."""
    img  = Image.fromarray(img_array.astype(np.uint8)).resize((64, 64))
    x    = np.array(img).reshape(-1, 1) / 255.0
    prob, _ = forward(x, params)
    prob    = float(prob[0, 0])
    is_cat  = prob > 0.5

    fig, ax = plt.subplots(figsize=(4, 4))
    fig.patch.set_facecolor('#0f0f1a')
    ax.imshow(img)
    ax.axis('off')
    verdict = 'CAT' if is_cat else 'NOT A CAT'
    color   = '#4ade80' if is_cat else '#f87171'
    ax.set_title(f'{verdict}\nConfidence: {prob:.1%}', color=color, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return is_cat, prob


print('📸 Upload an image to test Cat Detector 9000!')
try:
    uploaded = files.upload()
    for fname, content in uploaded.items():
        img = np.array(Image.open(io.BytesIO(content)).convert('RGB'))
        print(f'Analyzing: {fname}')
        predict_single(img, params)
except Exception:
    # fallback: pick a random test image if no file is uploaded
    print('No file uploaded. Testing on a random test image...')
    idx = np.random.randint(0, test_x_orig.shape[0])
    predict_single(test_x_orig[idx], params)
    print('Actual label:', 'CAT' if test_y[idx] == 1 else 'NOT CAT')

---
## 9. Hyperparameter Experiments

In [ ]:
# compare different learning rates — 1000 iterations each
print('Learning Rate Comparison')
print('=' * 40)
for lr in [0.01, 0.005, 0.001]:
    p = init_params([train_x.shape[0], 64, 32, 1])
    for _ in range(1000):
        AL, cache = forward(train_x, p)
        p = update(p, backward(AL, train_y, p, cache), lr)
    AL_t, _ = forward(test_x, p)
    print(f'  LR={lr:.4f} -> Test Accuracy: {accuracy(AL_t, test_y):.1f}%')

# compare different network architectures — 1000 iterations each
print('\nArchitecture Comparison')
print('=' * 40)
archs = [
    ([train_x.shape[0], 32, 1],          'Small  [32->1]'),
    ([train_x.shape[0], 64, 32, 1],      'Medium [64->32->1]'),
    ([train_x.shape[0], 128, 64, 1],     'Large  [128->64->1]'),
    ([train_x.shape[0], 64, 32, 16, 1],  'Deep   [64->32->16->1]'),
]
for arch, name in archs:
    p = init_params(arch)
    for _ in range(1000):
        AL, cache = forward(train_x, p)
        p = update(p, backward(AL, train_y, p, cache), 0.0075)
    AL_t, _  = forward(test_x, p)
    n_params = sum(p[k].size for k in p if k.startswith('W'))
    print(f'  {name} | {n_params:7,} params | Test: {accuracy(AL_t, test_y):.1f}%')